# 04 — Airport operations dashboard (Gold consumer)

This notebook is the **consumer** of the Gold layer: it answers the project question — *how busy is each European
airport, how are its runways used and how long do ground phases take, by day and by hour?* — from the four Gold
tables only, without any join or heavy computation (that is the purpose of Gold). The Gold tables are small:
they are read with Spark and converted to pandas for plotting.

Showcase airports: Toulouse-Blagnac (**LFBO**) and Stavanger (**ENZV**). All times are UTC.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))      # projet_opdi/src
import opdi_pipeline                                     # first: on the VM, findspark points to Spark 3.5 (/opt/spark)
from opdi_pipeline.config import load_settings
from opdi_pipeline.spark import get_spark

ENV = "prod"                     # "prod": full scope (~/opdi-data) | "dev": 10-day sample (~/opdi-data-dev)
settings = load_settings(ENV)
spark = get_spark(settings)
print(f"{ENV}: scope [{settings.start}, {settings.end}[ | data {settings.data_root} | Spark {spark.version}")

## Is Gold ready?

Consumers only read Gold when the `_SUCCESS` marker exists: it is written by the publication step, only if every validation check passed.

In [ ]:
marker = settings.gold_dir / "_SUCCESS"
assert marker.exists(), "Gold is not published: the last run did not pass its quality checks"
print(marker.read_text().strip())
status = spark.read.parquet(str(settings.gold_dir / "run_status")).toPandas()
status.sort_values("published_at_utc").tail(4)[["run_id", "table_name", "row_count", "size_mb", "published_at_utc"]]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

g = settings.gold_dir
daily = spark.read.parquet(str(g / "airport_daily_ops")).toPandas()
hourly = spark.read.parquet(str(g / "airport_hourly_ops")).toPandas()
runways = spark.read.parquet(str(g / "runway_usage")).toPandas()
ground = spark.read.parquet(str(g / "airport_ground_times")).toPandas()
for name, df in [("daily", daily), ("hourly", hourly), ("runways", runways), ("ground", ground)]:
    print(f"{name:8s} {len(df):>8,} rows")
SHOWCASE = ["LFBO", "ENZV"]

## 1. The busiest airports

In [ ]:
top = (daily.groupby(["airport_icao", "airport_name"], as_index=False)
             .agg(movements=("movements", "sum"), days=("movement_date", "nunique"))
             .assign(per_day=lambda d: d.movements / d.days)
             .nlargest(15, "per_day"))
ax = top.sort_values("per_day").plot.barh(x="airport_icao", y="per_day", legend=False, figsize=(8, 5))
ax.set_xlabel("movements per day (departures + arrivals)")
ax.set_title("15 busiest airports of the period")
plt.tight_layout()
top[["airport_icao", "airport_name", "movements", "per_day"]].round(0)

## 2. Daily traffic of the showcase airports

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
for ax, apt in zip(axes, SHOWCASE):
    d = daily[daily.airport_icao == apt].sort_values("movement_date")
    ax.plot(d.movement_date, d.departures, label="departures")
    ax.plot(d.movement_date, d.arrivals, label="arrivals")
    ax.set_title(f"{apt} — {d.airport_name.iloc[0] if len(d) else ''}")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
plt.tight_layout()
daily[daily.airport_icao.isin(SHOWCASE)].groupby("airport_icao").agg(
    movements_per_day=("movements", "mean"), peak_hour=("peak_hour_movements", "max"),
    operators=("distinct_operators", "mean"), pct_jet=("pct_jet", "mean"),
    pct_international=("pct_international", "mean"), pct_precise_time=("pct_time_from_event", "mean")).round(1)

## 3. Hourly profile (average movements per hour of the day, UTC)

In [ ]:
days = daily.groupby("airport_icao").movement_date.nunique()
profile = (hourly[hourly.airport_icao.isin(SHOWCASE)]
           .groupby(["airport_icao", "movement_hour"]).movements.sum().unstack(0).fillna(0)
           .div(days[SHOWCASE]).reindex(range(24), fill_value=0))
ax = profile.plot.bar(figsize=(12, 4))
ax.set_xlabel("hour of the day (UTC)")
ax.set_ylabel("movements per hour (average)")
ax.set_title("When are the showcase airports busy?")
plt.tight_layout()

## 4. Runway usage

In [ ]:
rw = (runways[runways.airport_icao.isin(SHOWCASE)]
      .groupby(["airport_icao", "runway_ident"], as_index=False)
      .agg(departures=("runway_departures", "sum"), arrivals=("runway_arrivals", "sum"),
           movements=("runway_movements", "sum"), length_ft=("length_ft", "first"), surface=("surface", "first")))
rw["share_pct"] = (rw.movements / rw.groupby("airport_icao").movements.transform("sum") * 100).round(1)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, apt in zip(axes, SHOWCASE):
    r = rw[rw.airport_icao == apt].set_index("runway_ident")[["departures", "arrivals"]]
    if len(r):
        r.plot.bar(stacked=True, ax=ax, title=f"{apt}: movements per runway")
plt.tight_layout()
rw.sort_values(["airport_icao", "movements"], ascending=[True, False])

## 5. Taxi times (where parking positions are observed)

Taxi-out = parking → departure runway; taxi-in = arrival runway → parking. They only exist where the ground receivers see the aircraft at its parking position: the coverage says how representative a figure is, so the chart keeps airports with at least 100 measured departures.

In [ ]:
taxi = (ground.groupby(["airport_icao", "airport_name"], as_index=False)
              .agg(measured_out=("taxi_out_measured_flights", "sum"), departures=("total_departures", "sum"),
                   measured_in=("taxi_in_measured_flights", "sum"), arrivals=("total_arrivals", "sum"),
                   median_taxi_out=("median_taxi_out_min", "median"), median_taxi_in=("median_taxi_in_min", "median")))
taxi["coverage_out_pct"] = (taxi.measured_out / taxi.departures * 100).round(1)
taxi["coverage_in_pct"] = (taxi.measured_in / taxi.arrivals * 100).round(1)
best = taxi[taxi.measured_out >= 100].nlargest(10, "measured_out")
if len(best):
    ax = best.set_index("airport_icao")[["median_taxi_out", "median_taxi_in"]].plot.bar(figsize=(10, 4))
    ax.set_ylabel("minutes (median of the daily medians)")
    ax.set_title("Taxi-out and taxi-in at the best-covered airports")
    plt.tight_layout()
taxi.sort_values("measured_out", ascending=False).head(10).round(1)

## 6. Reading the figures

*To be written from the prod run: traffic levels and patterns of the showcase airports, runway configuration, taxi times, and the limits given by the coverage columns (ADS-B ground coverage, parking positions mapped in OpenStreetMap).*